In [90]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

import torch

from torch_geometric.data import HeteroData


In [77]:
metadata = {
    "nodes": {},
    "segments": {},
    "ways": {}
}

# Load data

In [78]:
PREPROCESS_ROOT = Path("../data/preprocess")
RAW_ROOT = Path("../data/raw")

In [79]:
train_df = pd.read_csv("../data/raw/train.csv")

In [80]:
nodes_df = pd.read_csv(PREPROCESS_ROOT / "nodes.csv")
segments_df = pd.read_csv(PREPROCESS_ROOT / "segments.csv")
ways_df = pd.read_csv(PREPROCESS_ROOT / "ways.csv")
nodes_segments_edges_df = pd.read_csv(
    PREPROCESS_ROOT / "nodes_segments_edges_df.csv"
)

In [81]:
static_node_features = np.load(PREPROCESS_ROOT / "static_nodes.npy")
static_segment_features = np.load(PREPROCESS_ROOT / "static_segments.npy")
static_way_features = np.load(PREPROCESS_ROOT / "static_ways.npy")

# Xây dựng đồ thị không đồng nhất

In [82]:
data = HeteroData()

## Tĩnh

### Node

In [83]:
data["node"].x = static_node_features
data["segment"].x = static_segment_features
data["way"].x = static_way_features

### Edge

In [84]:
segments_df.head()

,id,created_at,updated_at,s_node_id,e_node_id,length,street_id,max_velocity,street_level,name,...,type_residential,type_school,type_secondary,type_secondary_link,type_tertiary,type_tertiary_link,type_trunk,type_trunk_link,type_unclassified,type_university
0,0,2020-10-18T13:26:17.551Z,2020-10-18T13:26:17.551Z,663,512,0.116,0,NaN,4,Nguyễn Văn Bá,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
1,1,2020-10-18T13:26:17.581Z,2020-10-18T13:26:17.581Z,1179,5601,0.026,1,NaN,3,Đường số 5,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,2020-10-18T13:26:17.756Z,2020-10-18T13:26:17.756Z,377,8976,0.007,1,NaN,3,Đường số 5,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3,2020-10-18T13:26:17.768Z,2020-10-18T13:26:17.768Z,11094,6619,0.008,2,40.0,3,Châu Văn Liêm,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,4,2020-10-18T13:26:17.772Z,2020-10-18T13:26:17.772Z,840,8429,0.043,3,NaN,4,Lê Văn Thịnh,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [96]:
# segment <-> way
## way -> segment
contain_segments = (
    segments_df[["street_id", "id"]]
    .drop_duplicates()
    .to_numpy()
    .astype(int)
)
print(contain_segments.shape)

## segment -> way
part_of_way = (
    segments_df[["id", "street_id"]]
    .drop_duplicates()
    .to_numpy()
    .astype(int)
)
print(contain_segments.shape)


(10027, 2)
(10027, 2)


In [ ]:
#
starts_with = (
    segments_df[["id", "s_node_id"]]
    .drop_duplicates()
    .to_numpy()
    .astype(int)
)
print(starts_with.shape)

(10027, 2)


In [89]:
ends_with = (
    segments_df[["id", "e_node_id"]]
    .drop_duplicates()
    .to_numpy()
    .astype(int)
)
print(ends_with.shape)

(10027, 2)


In [91]:
data["way", "contains", "segment"].edge_index = (
    torch.tensor(
        contain_segments.T,
        dtype=torch.long
    )
)

In [92]:
data["segment", "starts_with", "node"].edge_index = (
    torch.tensor(
        starts_with.T,
        dtype=torch.long
    )
)

In [93]:
data["segment", "ends_with", "node"].edge_index = (
    torch.tensor(
        ends_with.T,
        dtype=torch.long
    )
)